# NanoHarness Technical Documentation

This notebook documents the current internal structure of NanoHarness for development work.

## Architecture Overview

NanoHarness is split into small modules:

- `main.py` runs startup checks, then launches the harness.
- `helpers/startup.py` creates the app directory, sessions directory, and starter config.
- `core/harness.py` owns app lifecycle, config, sessions, slash commands, and the main loop.
- `core/agent.py` owns model streaming, reasoning display, cancellation, permissions, and tool orchestration.
- `core/auditor.py` owns command risk classification and auditor-model explanations.
- `core/tui.py` owns prompt input and prompt history.
- `tools/tools.py` owns the Bash tool schema and command execution.
- `core/paths.py` owns app paths.

## Startup Flow

`main.py` calls `startup_checks()` before starting the app. This guarantees app state exists before runtime code needs it.

```python
from helpers.startup import startup_checks
from core.harness import run

if __name__ == "__main__":
    startup_checks()
    run()
```

## App Paths

Runtime files are not stored in the repository by default. `core/paths.py` defines:

```python
APP_DIR = os.environ.get("NANOHARNESS_HOME", os.path.expanduser("~/.nanoharness"))
CONFIG_PATH = os.environ.get("NANOHARNESS_CONFIG", os.path.join(APP_DIR, "config.json"))
SESSIONS_DIR = os.path.join(APP_DIR, "sessions")
```

Startup ensures the primary config exists.

## Config Generation

`helpers/startup.py` contains a hardcoded `REFERENCE_CONFIG` and `CONFIG_VERSION`. If `CONFIG_PATH` does not exist, startup writes that reference to disk. If config exists, startup merges missing fields from the reference config and updates `config_version` without pruning unknown user keys.

Invalid JSON or non-object config files are backed up to `config.json.invalid.<timestamp>` and regenerated from the reference config. The config includes model settings, auditor model, Bash limits, and tool policy.

## Harness Responsibilities

`core/harness.py` is the app wrapper around the agent. It handles:

- Loading and saving config.
- Session creation, loading, deletion, and listing.
- Slash command handling.
- API key management via `/apikey`.
- OpenAI-compatible client creation.
- Main prompt loop.
- Error recovery after failed model turns.

## Agent Responsibilities

`core/agent.py` is limited to agent behavior:

- System prompt.
- Policy lookup and permission prompts.
- Streaming Chat Completions handling.
- Reasoning delta extraction and cleanup.
- esc/ctrl-c cancellation during streaming.
- Streaming tool-call accumulation.
- Bash tool auditing, user approval, and execution orchestration.

## Bash Tool

NanoHarness currently exposes only one model tool: `bash`. The model does not receive separate file tools.

`tools/tools.py` defines the schema and `run_bash()`. The command is executed with:

```python
subprocess.run(
    command,
    shell=True,
    executable="/bin/bash",
    cwd=resolved_cwd,
    capture_output=True,
    text=True,
    timeout=timeout_seconds,
)
```

Output is returned as JSON with stdout/stderr truncation flags.

## Auditor Flow

Before Bash runs, `core/agent.py` calls `core/auditor.py`:

1. `classify_bash_risks(command)` adds local risk hints with regex patterns.
2. `explain_bash_command(...)` calls the configured `auditor_model`.
3. The user sees the auditor explanation, risk hints, command, cwd, timeout, and output cap.
4. The command runs only if the user approves.

The auditor prompt asks for at most 70 words. The limit is intentionally in the prompt rather than config.

## Streaming And Tool Calls

NanoHarness is stream-only. `stream_assistant_response()` consumes streamed chunks, prints content/reasoning as they arrive, and reconstructs tool calls from partial deltas.

Tool call fragments are accumulated by index until the assistant turn finishes. After that, harness executes each tool call and appends tool results to the conversation.

## TUI Input

`core/tui.py` uses `prompt_toolkit` for prompt editing and persistent history. History is stored in `APP_DIR/prompt_history.txt`.

This provides arrow-key navigation, editable input, and history recall without custom terminal editing code.